# Project 3: Deep Neural Networks for Locating Unbounded-Magnitude Errors in Noisy Codewords

## The Model

- **One final model per code:** It outputs one number per location, and location $j$ is
  reported when that number is positive. All decisions are made by the trained networks and the other layers are fixed
  matrix multiplications, the averaging over views and the decision threshold.
- **Inputs:** Codes 1 and 2 take $y / \Delta$. Codes 3 to 6 take the syndrome $s = yH^T / \Delta$, a fixed float64
  multiplication with the parity-check matrix $H$ of the code that removes the codeword exactly, seen from every
  location (one token per location).
- **Networks:** Transformers with one token per location. Self-attention lets every location compare its value with
  all the others, which is what decides whether it holds a UME.
- **Training:** Binary cross-entropy per location (target 1 at required and 0 at forbidden locations; optional
  locations are masked out, because either answer is correct there), AdamW with a warm-up and a cosine-decaying
  learning rate. Every batch of 4,096 codewords is drawn fresh from the distribution of the project description, on
  the GPU, so the networks never see the same codeword twice and cannot overfit. Codes 3 to 6 add an auxiliary loss
  on per-location estimates of $e_j / \Delta$.
- **Symmetry averaging (codes 3 to 6):** The codes are unchanged by negacyclic shifts, reflection and sign change, so
  the final model runs its network(s) on 8 transformed copies of the codeword ("views") and averages the outputs.
- **Final models:** For each code, the candidate with the fewest errors on 1,000,000 validation codewords, with its
  threshold folded in, saved in float64 so that its decisions are the same on any machine. The six final models are
  stored together in one file, `final_model_v4.keras`, which the final cell loads.

## The Final Model of Each Code

| Code | $n$ | $k$ | Network | Parameters | Training |
|---|---|---|---|---|---|
| 1 | 10 | 3 | transformer, width 128, 4 blocks, 4 heads | 0.8 M | 60 epochs, a quarter of every batch from harder variants of the distribution |
| 2 | 16 | 3 | as code 1 | 0.8 M | as code 1 |
| 3 | 16 | 8 | transformer, width 128, 6 blocks, 8 heads; 8 views | 1.2 M | 60 epochs |
| 4 | 20 | 10 | as code 3 | 1.2 M | 120 epochs |
| 5 | 40 | 20 | two transformers, widths 128 and 192, 6 blocks, 8 heads; 4 views each, outputs averaged | 1.2 M + 2.7 M | 150 and 90 epochs |
| 6 | 80 | 40 | transformer, width 192, 6 blocks, 8 heads; 8 views | 2.7 M | 150 epochs |

An epoch is 2,000 steps (codes 1 and 2) or 1,000 steps (codes 3 to 6) of 4,096 codewords. Sections 7 and 15 show how
the final models were chosen.

## Outline

1. Configuration.

Part I (codes 1 and 2):

2. The two codes and the noisy-codeword distribution.
3. The validation and test sets.
4. The network architectures.
5. Training.
6. Training curves.
7. Validation, test performance and the choice of the final network.
8. Large-scale test of the final networks.
9. The final models.

Part II (codes 3 to 6):

10. The four codes, their structure and their noisy-codeword distribution.
11. The validation and test sets.
12. The network architecture.
13. Training.
14. Training curves.
15. Validation and test performance.
16. The final models.

17. **Final cell:** load the trained model and decode the six test files.


In [ ]:
# Install dependencies (Google Colab only; elsewhere the project environment is used).

import sys

if "google.colab" in sys.modules:
    %pip install -q "numpy>=2,<3" "tensorflow==2.21.0" "keras==3.15.1" matplotlib


## 1. Configuration

The defaults are the configuration of the submitted run. Environment variables were used to train the networks as
separate batch jobs (one per version and code, with `P3_TRAIN=1`); the submitted notebook was then executed with
`P3_TRAIN=0`, which loads those trained networks and runs every evaluation.

| Setting | Default | Environment variable |
|---|---|---|
| Output directory | `artifacts/notebook_v4` | `P3_OUT` |
| Train the networks (otherwise load trained ones) | `1` | `P3_TRAIN` |
| Codes | `1,2,3,4,5,6` | `P3_CODES` |
| Versions to train or load (codes 1 and 2) | `V1,V2,V3` | `P3_VERSIONS` |
| Versions to train or load (codes 3 to 6) | `H2,H3` | `P3_VERSIONS36` |
| Epochs, steps per epoch, batch size (overrides, for quick tests) | from the version recipes (sections 4 and 12), 2000 (codes 1, 2) and 1000 (codes 3 to 6), 4096 | `P3_EPOCHS`, `P3_STEPS`, `P3_BATCH` |
| Size of the validation and test sets | 1,000,000 each | `P3_EVAL_SIZE` |
| Millions of codewords in the large-scale test (section 8) | 100 | `P3_STRESS_MILLIONS` |
| Millions of codewords per hard variant (section 7.1) | 10 | `P3_HARD_MILLIONS` |


In [ ]:
import json
import os
import pickle
import time
import warnings
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", message=".*shuffle=True.*")   # Keras notes that tf.data input is not reshuffled
tf.get_logger().setLevel("ERROR")                                # no retracing notices from the evaluation helpers

OUT = Path(os.environ.get("P3_OUT", "artifacts/notebook_v4"))
TRAIN = os.environ.get("P3_TRAIN", "1") == "1"
CODES_TO_RUN = [int(c) for c in os.environ.get("P3_CODES", "1,2,3,4,5,6").split(",")]
CODES_12 = [c for c in CODES_TO_RUN if c in (1, 2)]                  # Part I
CODES_36 = [c for c in CODES_TO_RUN if c in (3, 4, 5, 6)]            # Part II
VERSIONS_TO_RUN = os.environ.get("P3_VERSIONS", "V1,V2,V3").split(",")
VERSIONS36_TO_RUN = os.environ.get("P3_VERSIONS36", "H2,H3").split(",")
EPOCHS = int(os.environ["P3_EPOCHS"]) if "P3_EPOCHS" in os.environ else None
STEPS = int(os.environ.get("P3_STEPS", "2000"))
STEPS36 = int(os.environ.get("P3_STEPS", "1000"))
BATCH = int(os.environ.get("P3_BATCH", "4096"))
EVAL_SIZE = int(os.environ.get("P3_EVAL_SIZE", "1000000"))
STRESS_MILLIONS = int(os.environ.get("P3_STRESS_MILLIONS", "100"))
HARD_MILLIONS = int(os.environ.get("P3_HARD_MILLIONS", "10"))
NOTEBOOK_TAG = "v4"
OUT.mkdir(parents=True, exist_ok=True)

# Full float32 precision on GPUs (no TF32 matrix multiplication), and GPU memory allocated as needed.
tf.config.experimental.enable_tensor_float_32_execution(False)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
print(f"TensorFlow {tf.__version__}, NumPy {np.__version__} | GPUs: {tf.config.list_physical_devices('GPU')}")
print(f"Output {OUT} | train {TRAIN} | codes {CODES_TO_RUN} | versions {VERSIONS_TO_RUN}" f" (codes 1, 2), {VERSIONS36_TO_RUN} (codes 3 to 6)")


# Part I: Codes 1 and 2

## 2. The Two Codes and the Noisy-Codeword Distribution

Both codes have $k = 3$; $\phi = (1 + \sqrt 5) / 2$.

- **Code 1** ($n = 10$): the columns of $G_1$ are the ten axes of a regular dodecahedron (all of length $\sqrt 3$).
- **Code 2** ($n = 16$): four columns $(-1, -1, -1)^T$, then each unit vector four times.

$\delta$ is the average column length of $G$, $\Delta_1 = (6 + 2\sqrt 5)\,\delta_1$, $\Delta_2 = 8\,\delta_2$ and
$Y = 3\Delta$. A noisy codeword is $y = uG + \epsilon + e$ with $u \sim U[-1, 1]^k$, $\epsilon \sim U[-\delta, \delta]^n$,
and $e$ with $A \in \{3, 2, 1, 0\}$ nonzero entries (probabilities $1/2, 1/4, 1/8, 1/8$) at $A$ distinct, uniformly
chosen locations, each uniform on $[-Y, Y] \setminus \{0\}$.

`sample` draws codewords with NumPy in float64 (for the fixed evaluation sets) and `tf_sample` draws the same
distribution with TensorFlow on the GPU (for the training batches). `correct` implements the correctness rule and
`check_output` the output-format rules. The cell checks every constant against the decimals given in the project
description.


In [ ]:
PHI = (1 + np.sqrt(5)) / 2
_P = 1 / PHI

G1 = np.array([[1, 1, 1, -1, 0, 0, _P, -_P, PHI, -PHI],
               [1, 1, -1, 1, PHI, PHI, 0, 0, _P, _P],
               [1, -1, 1, 1, _P, -_P, PHI, PHI, 0, 0]], dtype=np.float64)
G2 = np.zeros((3, 16))
G2[:, :4] = -1
for _i in range(3):
    G2[_i, 4 + 4 * _i:8 + 4 * _i] = 1


def _code(G, multiple):
    delta = float(np.linalg.norm(G, axis=0).mean())   # average column length
    Delta = multiple * delta
    return dict(G=G, k=G.shape[0], n=G.shape[1], delta=delta, Delta=Delta, Y=3 * Delta)


CODES = {1: _code(G1, 6 + 2 * np.sqrt(5)), 2: _code(G2, 8.0)}
A_VALUES = np.array([3, 2, 1, 0])
A_PROBS = np.array([1 / 2, 1 / 4, 1 / 8, 1 / 8])


def sample(code, size, rng):
    """Draw `size` noisy codewords of one code; returns y (size, n) and the UME vector e (size, n), float64."""
    C = CODES[code]; n = C["n"]
    u = rng.uniform(-1, 1, (size, C["k"]))
    eps = rng.uniform(-C["delta"], C["delta"], (size, n))
    A = rng.choice(A_VALUES, size=size, p=A_PROBS)
    chosen = np.argsort(rng.random((size, n)), axis=1).argsort(axis=1) < A[:, None]   # A uniform distinct locations
    # Uniform on [-Y, Y] without zero: magnitude Y (1 - U) lies in (0, Y] for U in [0, 1), with an independent sign.
    values = C["Y"] * (1 - rng.random((size, n))) * rng.choice((-1.0, 1.0), (size, n))
    e = np.where(chosen, values, 0.0)
    return u @ C["G"] + eps + e, e


def correct(code, e, reported):
    """Per-sample correctness of boolean decisions `reported` (size, n): all required and no forbidden location."""
    required = np.abs(e) > CODES[code]["Delta"]
    return ~(required & ~reported).any(axis=1) & ~(reported & (e == 0)).any(axis=1)


def to_sets(reported):
    """Boolean decisions (size, n) -> list of sets of Python ints, the required output format."""
    return [set(int(j) for j in np.flatnonzero(row)) for row in reported]


def check_output(outputs, num_inputs, n):
    """Raise ValueError unless `outputs` is a list of `num_inputs` sets of ints in [0, n) (no bools)."""
    if type(outputs) is not list or len(outputs) != num_inputs:
        raise ValueError(f"expected a list of {num_inputs} sets, got {type(outputs).__name__} of length {len(outputs)}")
    for r, s in enumerate(outputs):
        if type(s) is not set:
            raise ValueError(f"output {r} is a {type(s).__name__}, not a set")
        for j in s:
            if type(j) is not int or not 0 <= j < n:
                raise ValueError(f"output {r} contains {j!r}, not an int location in [0, {n})")


def tf_sample(code, size):
    """The same distribution as sample(), drawn in TensorFlow (float32) for on-the-fly training batches."""
    C = CODES[code]; n = C["n"]
    G = tf.constant(C["G"], tf.float32)
    u = tf.random.uniform((size, C["k"]), -1.0, 1.0)
    eps = tf.random.uniform((size, n), -C["delta"], C["delta"])
    A = tf.gather(tf.constant(A_VALUES, tf.int32),
                  tf.random.categorical(tf.math.log(tf.constant(A_PROBS[None], tf.float32)), size)[0])
    ranks = tf.argsort(tf.argsort(tf.random.uniform((size, n)), axis=1), axis=1)
    chosen = ranks < A[:, None]
    sign = tf.where(tf.random.uniform((size, n)) < 0.5, -1.0, 1.0)
    values = C["Y"] * (1.0 - tf.random.uniform((size, n))) * sign
    e = tf.where(chosen, values, 0.0)
    return u @ G + eps + e, e


# Check the constants against the project description.
reference = {1: dict(delta=1.732050808, Delta=18.138271538, Y=54.414814613),
             2: dict(delta=1.183012702, Delta=9.464101615, Y=28.392304845)}
rounded_G1 = np.array([[1.00, 1.00, 1.00, -1.00, 0.00, 0.00, 0.62, -0.62, 1.62, -1.62],
                       [1.00, 1.00, -1.00, 1.00, 1.62, 1.62, 0.00, 0.00, 0.62, 0.62],
                       [1.00, -1.00, 1.00, 1.00, 0.62, -0.62, 1.62, 1.62, 0.00, 0.00]])
assert np.array_equal(np.round(G1, 2), rounded_G1), "G1 does not round to the matrix in the description"
for code, C in CODES.items():
    for name, value in reference[code].items():
        assert abs(C[name] - value) < 1e-8, (code, name, C[name], value)
    print(f"Code {code}: k = {C['k']}, n = {C['n']}, delta = {C['delta']:.9f}, Delta = {C['Delta']:.9f}, Y = {C['Y']:.9f}")
print("All constants match the project description.")


### 2.1 The provided example files

The provided examples (1,000 noisy codewords per code with one permitted decoding each) show the input and output
formats and they come without their UME vectors, so they cannot measure accuracy. The cell checks that the example outputs
pass `check_output`, and compares the distribution of the example inputs with codewords drawn by `sample`: the
quantiles of $|y_j|$ and of the largest $|y_j|$ per codeword should agree up to sampling error (1,000 examples).


In [ ]:
def find_file(name, folders=(".", "data")):
    """The first existing path for a provided file (the notebook folder or data/), or None."""
    return next((Path(f) / name for f in folders if (Path(f) / name).exists()), None)


class CompatUnpickler(pickle.Unpickler):
    # Pickles written with NumPy 2 reference numpy._core; NumPy 1.x only has numpy.core.
    def find_class(self, module, name):
        if int(np.__version__.split(".")[0]) < 2 and module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)


def load_pickle(path):
    with open(path, "rb") as file:
        return CompatUnpickler(file).load()


EXAMPLES = {}
for code in CODES:
    inputs = find_file(f"Noisy-Codeword-Examples-for-Code-{code}.pkl")
    outputs = find_file(f"Decode-results-examples-for-code-{code}.pkl")
    if inputs is None or outputs is None:
        print(f"Code {code}: example files not found (optional)"); continue
    X, S = np.asarray(load_pickle(inputs), dtype=np.float64), load_pickle(outputs)
    n = CODES[code]["n"]
    assert X.shape == (len(X), n) and np.isfinite(X).all()
    check_output(S, len(X), n)
    EXAMPLES[code] = (X, S)
    y, _ = sample(code, 200_000, np.random.default_rng([3600, code]))
    q = [0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
    print(f"Code {code}: {len(X)} examples, outputs pass the format check; mean set size {np.mean([len(s) for s in S]):.2f}")
    print(f"  |y_j| quantiles {q}:  examples {np.round(np.quantile(np.abs(X), q), 2).tolist()}")
    print(f"                        sampled  {np.round(np.quantile(np.abs(y), q), 2).tolist()}")
    print(f"  max_j |y_j| quantiles: examples {np.round(np.quantile(np.abs(X).max(1), q), 2).tolist()}")
    print(f"                         sampled  {np.round(np.quantile(np.abs(y).max(1), q), 2).tolist()}")


## 3. The Validation and Test Sets

One fixed set of 1,000,000 validation codewords and one of 1,000,000 test codewords per code, drawn with `sample`
from seeds used nowhere else. Training never uses them: it draws fresh batches with TensorFlow's random generator.
The validation set selects the best epoch, the decision threshold and the final version; the test set is used only
to report performance.


In [ ]:
EVAL = {}
for code in CODES_12:
    y_val, e_val = sample(code, EVAL_SIZE, np.random.default_rng([3601, code]))
    y_test, e_test = sample(code, EVAL_SIZE, np.random.default_rng([3602, code]))
    EVAL[code] = dict(y_val=y_val, e_val=e_val, y_test=y_test, e_test=e_test)
    C = CODES[code]
    A = (e_test != 0).sum(1)
    required = (np.abs(e_test) > C["Delta"]).sum(1)
    optional = ((e_test != 0) & (np.abs(e_test) <= C["Delta"])).sum(1)
    print(f"Code {code}: {len(y_val):,} validation and {len(y_test):,} test codewords | test: share with A = 0..3: "
          + " ".join(f"{(A == a).mean():.4f}" for a in range(4))
          + f" | mean required {required.mean():.3f}, optional {optional.mean():.3f} locations per codeword")


## 4. The Network Architectures

Both architectures take $y / \Delta$ and output $n$ logits.

- **Residual MLP (V1):** a dense embedding of the whole vector, residual blocks (layer normalisation, a dense layer
  of twice the width with GELU, a dense layer back to the width), and a dense output layer with $n$ logits.
- **Transformer (V2):** one token per location. Token $j$ is $y_j w_j + b_j$ followed by a GELU layer, where $w_j$
  and $b_j$ are learned per location ($b_j$ acts as a location embedding). Pre-normalised transformer blocks
  (multi-head self-attention between the $n$ tokens, and a feed-forward layer of four times the width) let every
  location compare its value with all the others, which is what decoding requires: whether $y_j$ is a UME depends on
  how well the other values fit a codeword. A shared dense layer gives each token its logit.


In [ ]:
VERSIONS = {
    # V1: residual MLP on the whole vector y.
    "V1": dict(arch="mlp", width=512, blocks=4, heads=0, lr=1e-3, epochs=60),
    # V2: transformer with one token per location (value embedding + learned location embedding).
    "V2": dict(arch="transformer", width=128, blocks=4, heads=4, lr=5e-4, epochs=60),
    # V3: V2 trained with a quarter of every batch drawn from the hard variants (tf_sample_hard).
    "V3": dict(arch="transformer", width=128, blocks=4, heads=4, lr=5e-4, epochs=60, hard=0.25),
}


def recipe(version):
    R = dict(VERSIONS[version])
    if EPOCHS is not None:
        R["epochs"] = EPOCHS
    return R


def build_network(code, arch, width, blocks, heads, **_):
    n = CODES[code]["n"]
    inp = tf.keras.Input(shape=(n,), name="y")
    x = tf.keras.layers.Rescaling(1 / CODES[code]["Delta"], name="scale")(inp)
    if arch == "mlp":
        h = tf.keras.layers.Dense(width, activation="gelu", name="embed")(x)
        for b in range(blocks):
            z = tf.keras.layers.LayerNormalization(name=f"block{b}_norm")(h)
            z = tf.keras.layers.Dense(2 * width, activation="gelu", name=f"block{b}_hidden")(z)
            h = tf.keras.layers.Add()([h, tf.keras.layers.Dense(width, name=f"block{b}_out")(z)])
        logits = tf.keras.layers.Dense(n, name="logits")(tf.keras.layers.LayerNormalization(name="final_norm")(h))
    else:
        # Token j = y_j * w_j + b_j: a location-specific embedding of the value; b_j acts as a learned location
        # embedding. A per-token feed-forward layer then makes the embedding nonlinear in y_j.
        h = tf.keras.layers.EinsumDense("bn,nd->bnd", output_shape=(n, width), bias_axes="nd", name="embed")(x)
        h = tf.keras.layers.Dense(width, activation="gelu", name="embed_mlp")(h)
        for b in range(blocks):
            z = tf.keras.layers.LayerNormalization(name=f"block{b}_attention_norm")(h)
            z = tf.keras.layers.MultiHeadAttention(heads, width // heads, name=f"block{b}_attention")(z, z)
            h = tf.keras.layers.Add()([h, z])
            z = tf.keras.layers.LayerNormalization(name=f"block{b}_ffn_norm")(h)
            z = tf.keras.layers.Dense(4 * width, activation="gelu", name=f"block{b}_ffn_hidden")(z)
            h = tf.keras.layers.Add()([h, tf.keras.layers.Dense(width, name=f"block{b}_ffn_out")(z)])
        z = tf.keras.layers.Dense(1, name="token_logit")(tf.keras.layers.LayerNormalization(name="final_norm")(h))
        logits = tf.keras.layers.Reshape((n,), name="logits")(z)
    return tf.keras.Model(inp, logits, name=f"ume_locator_code{code}_{arch}")


for version in VERSIONS:
    for code in CODES:
        print(f"{version} code {code}: {build_network(code, **recipe(version)).count_params():,} parameters")


## 5. Training

- **Loss** (`masked_bce`): binary cross-entropy per location with target 1 at required and 0 at forbidden locations,
  averaged over the non-optional locations. The training labels are passed as the targets followed by the mask.
- **Data** (`training_batch`): every batch of 4,096 codewords is drawn anew by `tf_sample` inside a `tf.data`
  pipeline on the GPU.
- **Optimiser:** AdamW (weight decay 1e-5, $\beta_2 = 0.95$, gradients clipped to norm 1), learning rate warming up
  linearly over 2,000 steps from 1% of its peak, then decaying along a cosine to 1% of the peak.
- **Validation** (`Validate`): after every epoch, the network decodes the full validation set (logit > 0); the epoch
  with the fewest wrongly decoded codewords (ties: lower validation loss) is saved.
- Each network is trained with the random seed 636 + code.

### 5.1 Hard variants of the training data (V3)

The earlier versions decode the exact distribution without errors (sections 7 and 8), but tests on harder variants of
the distribution showed that their decision margins shrink, and some codewords are decoded wrongly, when the UMEs
lie close to $\Delta$ or when $u$ and $\epsilon$ sit at corners of their ranges. V3 therefore draws a quarter of
every training batch from three hard variants (`tf_sample_hard`, one variant per codeword, equally likely):

- **near:** always $A = 3$ UMEs, each magnitude uniform on $[0.7\Delta, 1.3\Delta]$, so required locations sit just
  above $\Delta$ next to optional ones just below it;
- **corner:** the usual $A$ and $e$, with $u \in \{-1, 1\}^k$ and $\epsilon \in \{-\delta, \delta\}^n$;
- **near-corner:** both.

Only the training data change and for validation, the threshold and the test use the exact distribution.


In [ ]:
def make_masked_bce(n):
    def masked_bce(y_true, logits):
        target, mask = y_true[:, :n], y_true[:, n:]
        loss = tf.nn.sigmoid_cross_entropy_with_logits(labels=target, logits=logits)
        return tf.reduce_sum(loss * mask) / tf.maximum(tf.reduce_sum(mask), 1.0)
    return masked_bce


def tf_sample_hard(code, size):
    """Training-only hard variants of the distribution, one of three per codeword (equally likely):
    near (A = 3, every UME magnitude uniform in [0.7, 1.3] Delta), corner (the usual A and e, with u in {-1, 1}^k and
    eps in {-delta, delta}^n), and near-corner (both)."""
    C = CODES[code]; n, k = C["n"], C["k"]
    variant = tf.random.uniform((size, 1), 0, 3, tf.int32)
    near, corner = variant != 2, variant != 0                         # 0 near, 1 near-corner, 2 corner
    sign = lambda shape: tf.where(tf.random.uniform(shape) < 0.5, -1.0, 1.0)
    u = tf.where(corner, sign((size, k)), tf.random.uniform((size, k), -1.0, 1.0))
    eps = tf.where(corner, C["delta"] * sign((size, n)), tf.random.uniform((size, n), -C["delta"], C["delta"]))
    A = tf.gather(tf.constant(A_VALUES, tf.int32),
                  tf.random.categorical(tf.math.log(tf.constant(A_PROBS[None], tf.float32)), size)[0])[:, None]
    A = tf.where(near, 3, A)
    mags = tf.where(near, C["Delta"] * tf.random.uniform((size, n), 0.7, 1.3),
                    C["Y"] * (1.0 - tf.random.uniform((size, n))))
    chosen = tf.argsort(tf.argsort(tf.random.uniform((size, n)), axis=1), axis=1) < A
    e = tf.where(chosen, mags * sign((size, n)), 0.0)
    return u @ tf.constant(C["G"], tf.float32) + eps + e, e


def make_training_batch(code, hard_fraction=0.0):
    Delta = CODES[code]["Delta"]
    hard = int(round(hard_fraction * BATCH))

    def training_batch(_):
        y, e = tf_sample(code, BATCH - hard)
        if hard:
            yh, eh = tf_sample_hard(code, hard)
            y, e = tf.concat([y, yh], 0), tf.concat([e, eh], 0)
        required = tf.cast(tf.abs(e) > Delta, tf.float32)
        optional = tf.cast((e != 0) & (tf.abs(e) <= Delta), tf.float32)
        return y, tf.concat([required, 1.0 - optional], axis=1)
    return training_batch


def predict_logits(network, y):
    f = tf.function(lambda b: network(b, training=False), jit_compile=True)
    return np.concatenate([np.asarray(f(tf.constant(y[s:s + 16384], tf.float32))) for s in range(0, len(y), 16384)])


def bce_np(code, logits, e):
    required = np.abs(e) > CODES[code]["Delta"]; mask = ~((e != 0) & ~required)
    loss = np.maximum(logits, 0) - logits * required + np.log1p(np.exp(-np.abs(logits)))
    return float((loss * mask).sum() / mask.sum())


class Validate(tf.keras.callbacks.Callback):
    """Full-validation-set evaluation after every epoch; keeps the epoch with the fewest wrong codewords."""

    def __init__(self, code, y, e, path):
        super().__init__(); self.code, self.y, self.e, self.path = code, y, e, path
        self.best = (np.inf, np.inf); self.history = []; self.start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logits = predict_logits(self.model, self.y)
        errors = int((~correct(self.code, self.e, logits > 0)).sum()); loss = bce_np(self.code, logits, self.e)
        lr = float(np.asarray(self.model.optimizer.learning_rate))
        row = dict(epoch=epoch + 1, loss=float(logs["loss"]), val_loss=loss, val_errors=errors, lr=lr,
                   minutes=(time.time() - self.start) / 60)
        self.history.append(row)
        improved = (errors, loss) < self.best
        if improved:
            self.best = (errors, loss); self.model.save(self.path)
        print(f"epoch {epoch + 1:3d}  loss {row['loss']:.3e}  val_loss {loss:.3e}  val_errors {errors:6d} "
              f"({errors / len(self.y):.2e})  lr {lr:.2e}  {row['minutes']:.1f} min{'  *' if improved else ''}", flush=True)


def model_path(version, code):
    return OUT / f"model_{version}_code{code}.keras"


def train_network(version, code):
    R = recipe(version); n = CODES[code]["n"]
    tf.keras.utils.set_random_seed(636 + code)
    network = build_network(code, **R)
    steps = R["epochs"] * STEPS; warmup = min(2000, steps // 10)
    lr = tf.keras.optimizers.schedules.CosineDecay(R["lr"] * 0.01, steps - warmup, alpha=0.01, warmup_target=R["lr"],
                                                   warmup_steps=warmup)
    network.compile(optimizer=tf.keras.optimizers.AdamW(lr, weight_decay=1e-5, beta_2=0.95, clipnorm=1.0),
                    loss=make_masked_bce(n))
    print(f"{version} code {code}: {R} | {STEPS} steps x {BATCH} codewords per epoch", flush=True)
    data = tf.data.Dataset.range(1).repeat().map(make_training_batch(code, R.get("hard", 0.0)),
                                                 num_parallel_calls=tf.data.AUTOTUNE).prefetch(8)
    val = Validate(code, EVAL[code]["y_val"], EVAL[code]["e_val"], model_path(version, code))
    network.fit(data, epochs=R["epochs"], steps_per_epoch=STEPS, callbacks=[val, tf.keras.callbacks.TerminateOnNaN()],
                verbose=0)
    history = dict(recipe=R, steps=STEPS, batch=BATCH, epochs=val.history)
    (OUT / f"history_{version}_code{code}.json").write_text(json.dumps(history, indent=1))


NETWORKS = {}
for version in VERSIONS_TO_RUN:
    for code in CODES_12:
        if TRAIN:
            train_network(version, code)
        if model_path(version, code).exists():
            NETWORKS[version, code] = tf.keras.models.load_model(model_path(version, code), compile=False)
            print(f"Loaded {model_path(version, code)}")
        else:
            print(f"{model_path(version, code)} not found: train it with P3_TRAIN=1")


## 6. Training Curves

Training loss (on the current batches), validation loss and the number of wrongly decoded validation codewords
(out of 1,000,000) after every epoch. Validation errors of zero cannot be drawn on the logarithmic scale; those epochs
are shown at the bottom edge of the plot.


In [ ]:
HISTORIES = {}
for version in VERSIONS_TO_RUN:
    for code in CODES_12:
        path = OUT / f"history_{version}_code{code}.json"
        if path.exists():
            HISTORIES[version, code] = json.loads(path.read_text())["epochs"]

if HISTORIES:
    fig, axes = plt.subplots(len(CODES_12), 2, figsize=(13, 4.2 * len(CODES_12)), squeeze=False)
    for row, code in enumerate(CODES_12):
        for version in VERSIONS_TO_RUN:
            if (version, code) not in HISTORIES:
                continue
            h = HISTORIES[version, code]; epochs = [r["epoch"] for r in h]
            line, = axes[row, 0].plot(epochs, [r["loss"] for r in h], "--", alpha=0.7, label=f"{version} training")
            axes[row, 0].plot(epochs, [r["val_loss"] for r in h], color=line.get_color(), label=f"{version} validation")
            axes[row, 1].plot(epochs, [max(r["val_errors"], 0.5) for r in h], "o-", ms=3, color=line.get_color(), label=version)
        axes[row, 0].set(yscale="log", xlabel="epoch", ylabel="masked BCE loss", title=f"Code {code}: loss")
        axes[row, 1].set(yscale="log", xlabel="epoch", ylabel="wrong codewords (of 1M)", ylim=(0.4, None),
                         title=f"Code {code}: validation errors (0 drawn at the bottom edge)")
        for ax in axes[row]:
            ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()
    for (version, code), h in HISTORIES.items():
        best = min(h, key=lambda r: (r["val_errors"], r["val_loss"]))
        first0 = next((r["epoch"] for r in h if r["val_errors"] == 0), None)
        print(f"{version} code {code}: {len(h)} epochs in {h[-1]['minutes']:.0f} min; best epoch {best['epoch']} "
              f"(val errors {best['val_errors']}, val loss {best['val_loss']:.2e}); first epoch with 0 val errors: {first0}")


## 7. Validation, Test Performance and the Choice of the Final Network

For every trained network:

- **Threshold:** location $j$ is reported when its logit exceeds a threshold $t$, chosen on the validation set as
  the middle of the range of thresholds (from $-8$ to $8$ in steps of 0.1) with the fewest wrongly decoded codewords.
- **Decision margin:** for one codeword, the smallest distance of a logit from $t$ on the correct side, over its
  required locations (logit $- t$) and forbidden locations ($t -$ logit); optional locations do not count. The codeword
  is decoded correctly exactly when its margin is positive. When the networks make no errors, the low end of the
  margin distribution shows how far they are from making one.
- **Choice** (section 7.1): for each code, the final network is the version with the fewest validation errors;
  ties are broken by the fewest errors on the hard validation variants, then by the larger 10th-smallest margin on them.
- **Test:** the test set is decoded once, with the chosen threshold.


In [ ]:
def margins(code, e, logits, t):
    """Per-codeword decision margin: min over required (logit - t) and forbidden (t - logit) locations."""
    required = np.abs(e) > CODES[code]["Delta"]
    per_location = np.where(required, logits - t, np.where(e == 0, t - logits, np.inf))
    return per_location.min(axis=1)


GRID = np.round(np.linspace(-8, 8, 161), 2)
RESULTS = {}
for (version, code), network in NETWORKS.items():
    ev, et = EVAL[code]["e_val"], EVAL[code]["e_test"]
    lv, lt = predict_logits(network, EVAL[code]["y_val"]), predict_logits(network, EVAL[code]["y_test"])
    errs = np.array([(~correct(code, ev, lv > t)).sum() for t in GRID])
    best = GRID[errs == errs.min()]; t = float(best[len(best) // 2])
    rep = lt > t; ok = correct(code, et, rep); A = (et != 0).sum(1)
    mv = np.sort(margins(code, ev, lv, t)); mt = np.sort(margins(code, et, lt, t))
    RESULTS[version, code] = dict(
        version=version, code=code, parameters=network.count_params(), threshold=t,
        zero_error_range=(float(best.min()), float(best.max())) if errs.min() == 0 else None,
        val_errors=int(errs.min()), val_margin_10th=float(mv[9]), test_errors=int((~ok).sum()),
        test_accuracy=float(ok.mean()), test_margin_10th=float(mt[9]),
        test_missed_required=int(((np.abs(et) > CODES[code]["Delta"]) & ~rep).any(1).sum()),
        test_forbidden_reported=int((rep & (et == 0)).any(1).sum()),
        test_accuracy_by_A=[float(ok[A == a].mean()) for a in range(4)])

print(f"{'':12s} {'params':>9s} {'threshold':>9s} {'0-error thresholds':>19s} {'val errors':>10s} {'val margin':>10s} "
      f"{'test errors':>11s} {'test accuracy':>13s} {'test margin':>11s}   test accuracy for A = 0, 1, 2, 3")
for (version, code), r in sorted(RESULTS.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    zr = "none" if r["zero_error_range"] is None else f"{r['zero_error_range'][0]:g} to {r['zero_error_range'][1]:g}"
    print(f"{version} code {code}  {r['parameters']:9,d} {r['threshold']:9.2f} {zr:>19s} {r['val_errors']:10d} "
          f"{r['val_margin_10th']:10.2f} {r['test_errors']:11d} {r['test_accuracy']:13.6f} {r['test_margin_10th']:11.2f}   "
          + ", ".join(f"{a:.6f}" for a in r["test_accuracy_by_A"]))
print("(margins: 10th-smallest per-codeword decision margin, in logits)")


### 7.1 Hard variants of the distribution (validation) and the choice

All versions decode the 1,000,000 validation and test codewords without errors, so the exact distribution alone
cannot separate them. This section decodes three harder variants of the distribution, the same ones V3 partly trains
on (section 5.1): **near** (always 3 UMEs, magnitudes uniform on $[0.7\Delta, 1.3\Delta]$), **near-corner** (as near,
with $u$ and $\epsilon$ at random corners of their ranges) and **corner** (the usual UMEs, with $u$ and $\epsilon$ at
corners). For each variant, 10 million codewords are drawn with NumPy from validation seeds used nowhere else, and
every network decodes them with its threshold from section 7. These codewords serve evaluation only; they are never
part of the decoding. A brute-force decoder that is told the true UME locations decodes such codewords correctly, so
every error here is an error of the network, not an ambiguous codeword.


In [ ]:
VARIANTS = ("near", "near-corner", "corner")


def draw_variant(code, size, rng, variant):
    """Evaluation-only hard variants (NumPy, float64), the same distributions as tf_sample_hard's."""
    C = CODES[code]; n, k = C["n"], C["k"]
    corner = variant.endswith("corner")
    u = rng.choice((-1.0, 1.0), (size, k)) if corner else rng.uniform(-1, 1, (size, k))
    eps = C["delta"] * rng.choice((-1.0, 1.0), (size, n)) if corner else rng.uniform(-C["delta"], C["delta"], (size, n))
    if variant.startswith("near"):
        A = np.full(size, 3)
        mags = C["Delta"] * rng.uniform(0.7, 1.3, (size, n))
    else:
        A = rng.choice(A_VALUES, size=size, p=A_PROBS)
        mags = C["Y"] * (1 - rng.random((size, n)))
    chosen = np.argsort(rng.random((size, n)), axis=1).argsort(axis=1) < A[:, None]
    e = np.where(chosen, mags * rng.choice((-1.0, 1.0), (size, n)), 0.0)
    return u @ C["G"] + eps + e, e


HARD_RESULTS = {}
for code in CODES_12:
    nets = [(v, NETWORKS[v, code], RESULTS[v, code]["threshold"]) for v in VERSIONS_TO_RUN if (v, code) in NETWORKS]
    fs = {v: tf.function(lambda b, net=net: net(b, training=False), jit_compile=True) for v, net, _ in nets}
    for v, _, _ in nets:
        HARD_RESULTS[v, code] = dict(wrong={x: 0 for x in VARIANTS}, lowest=np.full(10, np.inf))
    for index, variant in enumerate(VARIANTS, start=1):
        for chunk in range(HARD_MILLIONS):
            y, e = draw_variant(code, 1_000_000, np.random.default_rng([3603, code, index, chunk]), variant)
            for v, _, t in nets:
                logits = np.concatenate([np.asarray(fs[v](tf.constant(y[s:s + 65536], tf.float32)))
                                         for s in range(0, len(y), 65536)])
                h = HARD_RESULTS[v, code]
                h["wrong"][variant] += int((~correct(code, e, logits > t)).sum())
                h["lowest"] = np.sort(np.concatenate([h["lowest"], np.sort(margins(code, e, logits, t))[:10]]))[:10]

print(f"Wrongly decoded codewords out of {HARD_MILLIONS} million per variant (validation seeds), "
      "and the 10th-smallest margin over all of them:")
print(f"{'':12s} " + " ".join(f"{x:>12s}" for x in VARIANTS) + f" {'total':>8s} {'margin':>8s}")
for (v, code), h in sorted(HARD_RESULTS.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    h["total"] = sum(h["wrong"].values()); h["margin_10th"] = float(h["lowest"][9])
    print(f"{v} code {code}  " + " ".join(f"{h['wrong'][x]:12d}" for x in VARIANTS) + f" {h['total']:8d} {h['margin_10th']:8.2f}")

FINAL = {}
for code in CODES_12:
    candidates = [r for (v, c), r in RESULTS.items() if c == code]
    if candidates:
        FINAL[code] = min(candidates, key=lambda r: (r["val_errors"], HARD_RESULTS[r["version"], code]["total"],
                                                     -HARD_RESULTS[r["version"], code]["margin_10th"]))
        print(f"Code {code}: final network {FINAL[code]['version']} (threshold {FINAL[code]['threshold']:g})")


## 8. Large-Scale Test of the Final Networks

With no errors on 1,000,000 test codewords, the test set only shows that the error rate is below about $3 \times
10^{-6}$ (95% confidence). This section decodes 100 million further fresh test codewords per code (seeded chunks of
1,000,000, drawn with `sample` from seeds used nowhere else) with each final network and its threshold, counts the
wrongly decoded codewords, and records the decision margins. With $W$ errors in $N$ codewords, the error rate is
below $3/N$ when $W = 0$ (95% confidence).


In [ ]:
STRESS = {}
for code, r in FINAL.items() if STRESS_MILLIONS > 0 else []:   # P3_STRESS_MILLIONS=0 skips this test
    network, t = NETWORKS[r["version"], code], r["threshold"]
    f = tf.function(lambda b, network=network: network(b, training=False), jit_compile=True)
    wrong, lowest, start = 0, np.full(10, np.inf), time.time()
    for chunk in range(STRESS_MILLIONS):
        y, e = sample(code, 1_000_000, np.random.default_rng([3604, code, chunk]))
        logits = np.concatenate([np.asarray(f(tf.constant(y[s:s + 65536], tf.float32))) for s in range(0, len(y), 65536)])
        wrong += int((~correct(code, e, logits > t)).sum())
        lowest = np.sort(np.concatenate([lowest, np.sort(margins(code, e, logits, t))[:10]]))[:10]
    STRESS[code] = dict(version=r["version"], codewords=STRESS_MILLIONS * 1_000_000, wrong=wrong,
                        lowest_margins=lowest.round(2).tolist())
    bound = f"error rate below {3 / (STRESS_MILLIONS * 1e6):.0e} (95%)" if wrong == 0 else f"error rate {wrong / (STRESS_MILLIONS * 1e6):.2e}"
    print(f"Code {code} ({r['version']}): {wrong} wrong of {STRESS_MILLIONS:,} million codewords -> {bound}; "
          f"lowest margins {lowest.round(2).tolist()} ({(time.time() - start) / 60:.1f} min)")


## 9. The Final Models

Each final model is the chosen network with two changes that do not alter what it computes:

- **The threshold is folded in:** a final layer subtracts $t$ from the logits, so the final model reports location $j$
  exactly when its output is positive.
- **float64:** the network is rebuilt with float64 layers and the trained weights are copied into it. The
  submitted results are compared with the outputs of the model when it is run again; logits of optional
  locations can lie arbitrarily close to the threshold, where float32 rounding differences between machines (up to
  about $10^{-4}$) could flip a decision. In float64 these differences shrink to about $10^{-12}$.

The cell saves `final_model_v4_code1.keras` and `final_model_v4_code2.keras`, checks that the final models decode the
test set exactly like the selected networks, and runs them on the provided example inputs and on the first 100,000
test codewords written in the same format as the test files.


In [ ]:
def final_model_path(code):
    return OUT / f"final_model_{NOTEBOOK_TAG}_code{code}.keras"


def make_final_model(code, r):
    tf.keras.config.set_dtype_policy("float64")
    try:
        network = build_network(code, **recipe(r["version"]))
        inp = tf.keras.Input(shape=(CODES[code]["n"],), name="y")
        out = tf.keras.layers.Rescaling(1.0, offset=-r["threshold"], name="minus_threshold")(network(inp))
        final = tf.keras.Model(inp, out, name=f"final_ume_locator_code{code}")
    finally:
        tf.keras.config.set_dtype_policy("float32")
    network.set_weights([w.astype(np.float64) for w in NETWORKS[r["version"], code].get_weights()])
    return final


def decode(final, y):
    """What the final cell does: float64 logits on the CPU in batches of 4,096, location reported when positive."""
    with tf.device("/CPU:0"):
        out = np.concatenate([np.asarray(final(tf.constant(y[s:s + 4096], tf.float64), training=False))
                              for s in range(0, len(y), 4096)]) if len(y) else np.zeros((0, final.output_shape[-1]))
    return out > 0


for code, r in FINAL.items():
    final = make_final_model(code, r)
    final.save(final_model_path(code))
    final = tf.keras.models.load_model(final_model_path(code), compile=False)
    y, e = EVAL[code]["y_test"], EVAL[code]["e_test"]
    start = time.time(); rep64 = decode(final, y); seconds = time.time() - start
    rep32 = predict_logits(NETWORKS[r["version"], code], y) > r["threshold"]
    optional = (e != 0) & (np.abs(e) <= CODES[code]["Delta"])
    print(f"Code {code}: saved {final_model_path(code)} ({r['version']}, threshold {r['threshold']:g}, output dtype "
          f"{final.output.dtype}); test accuracy {correct(code, e, rep64).mean():.6f}; codewords decoded differently from "
          f"the float32 network: {(rep64 != rep32).any(1).sum()} (all at optional locations: {not (rep64 != rep32)[~optional].any()}); "
          f"CPU speed {len(y) / seconds:,.0f} codewords/s")
    if code in EXAMPLES:
        X, S = EXAMPLES[code]
        ours = to_sets(decode(final, X)); check_output(ours, len(X), CODES[code]["n"])
        print(f"  examples: identical to the example output {np.mean([a == b for a, b in zip(ours, S)]):.3f}, "
              f"subset of it {np.mean([a <= b for a, b in zip(ours, S)]):.3f}, superset {np.mean([a >= b for a, b in zip(ours, S)]):.3f} "
              "(differences are expected at optional locations)")
    heldout = OUT / f"Noisy-Codeword-Heldout-for-Code-{code}.pkl"
    with open(heldout, "wb") as file:
        pickle.dump([[float(v) for v in row] for row in y[:100_000]], file)
    X = np.asarray(load_pickle(heldout), dtype=np.float64)
    ours = to_sets(decode(final, X)); check_output(ours, len(X), CODES[code]["n"])
    reported = np.zeros((len(X), CODES[code]["n"]), bool)
    for row, s in enumerate(ours):
        reported[row, list(s)] = True
    print(f"  {heldout.name}: {len(ours):,} outputs pass the format check, accuracy {correct(code, e[:len(X)], reported).mean():.6f}")


# Part II: Codes 3 to 6

## 10. The Four Codes, Their Structure and Their Noisy-Codeword Distribution

The file `codes_3_to_6.pkl` provides $G$ and $\Delta$ for codes 3 to 6. `sample_harmonic` draws noisy codewords with
NumPy in float64 exactly as in the project description (rejection sampling, both sample groups, joint shuffle; for the
evaluation sets). `tf_sample_harmonic` draws the same distribution with TensorFlow in float64 on the GPU (truncated
normals by the inverse CDF) and returns the syndrome, for the training batches.

**Structure.** Every row of these four matrices is a cosine or sine of an *odd* harmonic $m = 1, 3, \dots, k-1$
sampled at $\theta_j = \pi j / n$ (scaled so that every column has length 1); the cell below checks this to $10^{-12}$.
The odd harmonics that $G$ leaves out, $m = k+1, \dots, n-1$, give a parity-check matrix $H$ ($r \times n$, $r = n - k$,
orthonormal rows, $GH^T = 0$). The **syndrome** $s = yH^T / \Delta$ therefore does not depend on $u$:
$s = (\epsilon + e)H^T / \Delta$. It is the input of the networks. It is one matrix multiplication, done in float64,
because $y$ is about $10^6$ times larger than the noise for code 6, and float32 rounding of $uG$ would swamp $\epsilon$.

**Location frames.** `ROTATIONS[j]` rotates every (cos, sin) pair of $s$ by $-m\theta_j$; then an error at location
$j$ alone gives the same pattern $(1, 0, 1, 0, \dots)$ in the frame of location $j$, wherever $j$ is, and the frame of
$j$ depends only on the error positions relative to $j$. The network receives $s$ in all $n$ frames (one token per
location; again a fixed matrix multiplication).

In [ ]:
HARMONIC_A = {3: 4, 4: 5, 5: 10, 6: 20}               # fixed number of UMEs per codeword
DESCRIPTION_36 = {3: (16, 8, 2773.9396039298235), 4: (20, 10, 25631.144050459065), 5: (40, 20, 32768.0), 6: (80, 40, 262144.0)}
codes_file = find_file("codes_3_to_6.pkl")
assert codes_file is not None, "codes_3_to_6.pkl (provided with the project) must be in the notebook folder or data/"
ENTRIES_36 = load_pickle(codes_file)


def harmonic_code(code, entry):
    G = np.asarray(entry["G"], dtype=np.float64); k, n = G.shape
    theta = np.pi * np.arange(n) / n
    # G = the odd harmonics 1, 3, ..., k-1 (a cos row and a sin row each) at theta_j, scaled to unit-length columns.
    harmonic = np.array([f(m * theta) for m in range(1, k, 2) for f in (np.cos, np.sin)]) / np.sqrt(k / 2)
    assert np.abs(G - harmonic).max() < 1e-12, f"code {code}: G is not the odd-harmonic matrix"
    # H = the odd harmonics k+1, ..., n-1 that G leaves out: orthonormal rows, G H^T = 0.
    freqs = np.arange(k + 1, n, 2)
    H = np.array([f(m * theta) for m in freqs for f in (np.cos, np.sin)]) / np.sqrt(n / 2)
    assert np.abs(G @ H.T).max() < 1e-12 and np.allclose(H @ H.T, np.eye(n - k))
    # ROTATIONS[j] (r x r): rotates every (cos, sin) pair of frequency m by -m theta_j, so ROTATIONS[j] @ H[:, j] = H[:, 0].
    rot = np.zeros((n, n - k, n - k))
    for p, m in enumerate(freqs):
        c, s = np.cos(m * theta), np.sin(m * theta)
        rot[:, 2 * p, 2 * p], rot[:, 2 * p, 2 * p + 1] = c, s
        rot[:, 2 * p + 1, 2 * p], rot[:, 2 * p + 1, 2 * p + 1] = -s, c
    assert np.allclose(np.einsum("jqr,rj->jq", rot, H), H[:, :1].T)
    delta = float(np.linalg.norm(G, axis=0).mean())          # average column length: 1
    return dict(G=G, k=k, n=n, r=n - k, delta=delta, Delta=float(entry["Delta"]), A=HARMONIC_A[code], H=H, ROTATIONS=rot)


for code in HARMONIC_A:
    CODES[code] = harmonic_code(code, ENTRIES_36[f"code_{code}"])


def truncated_normal(rng, mean, sd, shape, low, high, low_open):
    """Rejection sampling as described: draw N(mean, sd) and redraw values outside [low, high] ((low, high] if
    low_open) until all lie inside."""
    x = rng.normal(mean, sd, shape)
    bad = ~(((x > low) if low_open else (x >= low)) & (x <= high))
    while bad.any():
        x[bad] = rng.normal(mean, sd, int(bad.sum()))
        bad[bad] = ~(((x[bad] > low) if low_open else (x[bad] >= low)) & (x[bad] <= high))
    return x


def sample_harmonic(code, size, rng, return_group=False):
    """Codes 3-6: `size` noisy codewords exactly as described (both groups, shuffled jointly). Returns y and e
    (size, n), float64, and with return_group the group of every sample (1 = Group I, 2 = Group II)."""
    C = CODES[code]; n, k, A, D = C["n"], C["k"], C["A"], C["Delta"]
    u = rng.uniform(-2 * D, 2 * D, (size, k))
    eps = truncated_normal(rng, 1.0, 0.3, (size, n), 0.0, 1.0, False) * rng.choice((-1.0, 1.0), (size, n))
    locations = np.argsort(rng.random((size, n)), axis=1)[:, :A]         # A distinct, uniformly chosen locations
    n1 = (size + 1) // 2                                                 # ceil(size / 2) samples in Group I
    mags = np.empty((size, A))
    mags[:n1] = truncated_normal(rng, D, 0.3 * D, (n1, A), D, 2 * D, True)
    mags[n1:] = truncated_normal(rng, D, 0.3 * D, (size - n1, A), 0.0, 2 * D, True)
    redo = n1 + np.flatnonzero(mags[n1:].max(axis=1) <= D)               # Group II: none above Delta -> redraw all A
    while len(redo):
        mags[redo] = truncated_normal(rng, D, 0.3 * D, (len(redo), A), 0.0, 2 * D, True)
        redo = redo[mags[redo].max(axis=1) <= D]
    e = np.zeros((size, n))
    np.put_along_axis(e, locations, mags * rng.choice((-1.0, 1.0), (size, A)), axis=1)
    y = u @ C["G"] + eps + e
    group = np.where(np.arange(size) < n1, 1, 2)
    order = rng.permutation(size)                                        # joint shuffle of both groups
    return (y[order], e[order], group[order]) if return_group else (y[order], e[order])


def syndrome(code, y):
    """s = y H^T / Delta in float64 (one matrix multiplication); independent of u."""
    return np.asarray(y, np.float64) @ CODES[code]["H"].T / CODES[code]["Delta"]


def tf_sample_harmonic(code, size):
    """The same distribution in TensorFlow, everything in float64 (truncated normals by the inverse CDF, the same
    distribution as rejection sampling; Group II sets without a magnitude above Delta are redrawn). Returns the
    syndrome s = y H^T / Delta as float32 and e as float64, for the training batches."""
    C = CODES[code]; n, k, A, D = C["n"], C["k"], C["A"], C["Delta"]
    f64 = tf.float64
    phi = lambda z: 0.5 * tf.math.erfc(-z / np.sqrt(2.0))

    def truncated(shape, mean, sd, low, high):
        a, b = phi(tf.constant((low - mean) / sd, f64)), phi(tf.constant((high - mean) / sd, f64))
        p = b - tf.random.uniform(shape, dtype=f64) * (b - a)           # in (a, b]
        return tf.clip_by_value(mean + sd * tf.math.ndtri(p), low, high)

    sign = lambda shape: tf.where(tf.random.uniform(shape) < 0.5, tf.constant(-1.0, f64), tf.constant(1.0, f64))
    u = tf.random.uniform((size, k), -2 * D, 2 * D, dtype=f64)
    eps = truncated((size, n), 1.0, 0.3, 0.0, 1.0) * sign((size, n))
    locations = tf.argsort(tf.random.uniform((size, n)), axis=1)[:, :A]
    n1 = (size + 1) // 2
    mags1 = truncated((n1, A), D, 0.3 * D, D, 2 * D)
    mags1 = tf.where(mags1 > D, mags1, tf.constant(np.nextafter(D, 2 * D), f64))   # (Delta, 2 Delta]
    draw2 = lambda: truncated((size - n1, A), D, 0.3 * D, 0.0, 2 * D)
    mags2 = tf.while_loop(lambda m: tf.reduce_any(tf.reduce_max(m, 1) <= D),
                          lambda m: [tf.where(tf.reduce_max(m, 1, keepdims=True) <= D, draw2(), m)], [draw2()])[0]
    mags2 = tf.where(mags2 > 0, mags2, tf.constant(np.nextafter(0.0, 1.0), f64))
    values = tf.concat([mags1, mags2], 0) * sign((size, A))
    e = tf.einsum("ba,ban->bn", values, tf.one_hot(locations, n, dtype=f64))
    y = u @ tf.constant(C["G"], f64) + eps + e
    s = y @ tf.constant(C["H"].T / D, f64)
    return tf.cast(s, tf.float32), e


for code in HARMONIC_A:
    C = CODES[code]
    assert (C["n"], C["k"], C["Delta"]) == DESCRIPTION_36[code] and abs(C["delta"] - 1) < 1e-12
    print(f"Code {code}: n = {C['n']}, k = {C['k']}, r = {C['r']}, A = {C['A']}, delta = {C['delta']:.12f}, "
          f"Delta = {C['Delta']!r}; G = odd harmonics 1..{C['k'] - 1}, H = odd harmonics {C['k'] + 1}..{C['n'] - 1}")
print("All matrices, thresholds and UME counts match the project description.")


### 10.1 Checks of the sampler and the provided example files

For each code: 200,000 codewords from `sample_harmonic` (Group I share, magnitudes inside their intervals, at least
one required UME per Group II sample, the noise $s - eH^T/\Delta$ at the level of $\epsilon$), the same statistics from
`tf_sample_harmonic`, and a comparison with the 1,000 provided example inputs: quantiles of $|y_j|$ and of the
syndrome norm $\|s\|$. The example outputs pass `check_output`; half of them (Group I) have exactly $A$ locations.


In [ ]:
for code in HARMONIC_A:
    C = CODES[code]; n, A, D = C["n"], C["A"], C["Delta"]
    y, e, g = sample_harmonic(code, 200_000, np.random.default_rng([3600, code]), return_group=True)
    mags = np.abs(e[e != 0]).reshape(len(e), A) / D
    s = syndrome(code, y)
    noise = s - e @ C["H"].T / D
    assert ((e != 0).sum(1) == A).all() and np.mean(g == 1) == 0.5
    assert (mags[g == 1] > 1).all() and (mags[g == 1] <= 2).all() and (mags[g == 2] > 0).all() and (mags <= 2).all()
    assert (mags[g == 2].max(1) > 1).all()
    s_tf, e_tf = tf_sample_harmonic(code, 200_000)
    m_tf = np.abs(e_tf.numpy()[e_tf.numpy() != 0]) / D
    print(f"Code {code}: Group II share with |e| > Delta: {np.mean(mags[g == 2] > 1):.4f} (TensorFlow sampler: "
          f"{2 * np.mean(m_tf > 1) - 1:.4f}); mean |e| / Delta {mags.mean():.4f} (TensorFlow {m_tf.mean():.4f}); "
          f"noise in s: std {noise.std() * D:.3f} / Delta")
    inputs = find_file(f"Noisy-Codeword-Examples-for-Code-{code}.pkl")
    outputs = find_file(f"Decode-results-examples-for-code-{code}.pkl")
    if inputs is None or outputs is None:
        print("  example files not found (optional)"); continue
    X, S = np.asarray(load_pickle(inputs), dtype=np.float64), load_pickle(outputs)
    assert X.shape == (len(X), n) and np.isfinite(X).all()
    check_output(S, len(X), n)
    EXAMPLES[code] = (X, S)
    q = [0.1, 0.5, 0.9, 0.99]
    print(f"  {len(X)} examples; outputs pass the format check; outputs with exactly A = {A} locations: "
          f"{np.mean([len(x) == A for x in S]):.3f}")
    print(f"  |y_j| / Delta quantiles {q}: examples {np.round(np.quantile(np.abs(X) / D, q), 3).tolist()}, "
          f"sampled {np.round(np.quantile(np.abs(y) / D, q), 3).tolist()}")
    print(f"  ||s|| quantiles:              examples {np.round(np.quantile(np.linalg.norm(syndrome(code, X), axis=1), q), 3).tolist()}, "
          f"sampled {np.round(np.quantile(np.linalg.norm(s, axis=1), q), 3).tolist()}")


## 11. The Validation and Test Sets (Codes 3 to 6)

As in Part I: one fixed set of 1,000,000 validation and one of 1,000,000 test codewords per code, drawn with
`sample_harmonic` (the exact two-group mixture) from seeds used nowhere else; training draws fresh batches with
TensorFlow. The networks receive the syndromes (computed in float64, passed as float32); the final models (section 16)
take $y$ itself.


In [ ]:
EVAL36 = {}
for code in CODES_36:
    entry = {}
    for split, seed in (("val", 3601), ("test", 3602)):
        y, e, g = sample_harmonic(code, EVAL_SIZE, np.random.default_rng([seed, code]), return_group=True)
        entry[f"s_{split}"], entry[f"e_{split}"], entry[f"g_{split}"] = syndrome(code, y).astype(np.float32), e, g
        if split == "test":
            entry["y_test"] = y
    EVAL36[code] = entry
    C = CODES[code]; e = entry["e_test"]
    required = (np.abs(e) > C["Delta"]).sum(1); optional = ((e != 0) & (np.abs(e) <= C["Delta"])).sum(1)
    print(f"Code {code}: {EVAL_SIZE:,} validation and {EVAL_SIZE:,} test codewords | test: Group I share "
          f"{np.mean(entry['g_test'] == 1):.3f}; mean required {required.mean():.3f}, optional {optional.mean():.3f} locations per codeword")


## 12. The Network Architecture (Codes 3 to 6)

Input: the syndrome $s$ ($r$ numbers, in units of $\Delta$). Output: $n$ logits ("report location $j$").

1. **Location tokens** (fixed, not trained): token $j$ = `ROTATIONS[j]` $s$, the syndrome in the frame of location $j$
   ($n$ tokens of $r$ numbers; one matrix multiplication). An isolated UME at $j$ shows up in token $j$ as the
   constant pattern $(1, 0, 1, 0, \dots) \cdot e_j \sqrt{2/n}$; UMEs elsewhere add rotated patterns that depend only
   on their offset from $j$.
2. **Embedding:** a shared linear layer to the width, a learned per-location bias (location embedding), and a GELU
   layer.
3. **Six transformer blocks** (pre-normalised multi-head self-attention between the $n$ tokens and a feed-forward layer
   of four times the width). Whether location $j$ holds a UME depends on how well the rest of the syndrome can be
   explained by UMEs elsewhere; attention lets every location compare its view with all the others, which is the
   "explaining away" that clusters of adjacent UMEs require.
4. **Heads:** a shared dense layer gives every token its logit. A second dense layer (the value head) estimates
   $e_j / \Delta$ at every location; it is used only for an auxiliary training loss (section 13), and the final
   models output only the logits.

**Versions:** H2 (width 128, about 1.2 million parameters) for all four codes; for codes 5 and 6, where the networks
are furthest from perfect decoding, also H3 (width 192, about 2.7 million parameters, trained for about as long as
H2). The number of training epochs grows with the difficulty of the code; the final model of each code is chosen on
the validation set (section 15).

**Development.** Shorter runs of the same code (`train36.py`; 200,000 validation codewords) compared this design
with two refinement networks that feed the syndrome left unexplained by the current value estimates (a fixed matrix
multiplication) back into later blocks, with fixed (R1) or growing, additive (R4) amplification. They were not better:

| Code 5, validation accuracy | epoch 7 | epoch 11 | epoch 17 |
|---|---|---|---|
| H2 (45-epoch schedule) | 0.776 | 0.881 | 0.935 |
| R1 refinement (45-epoch schedule) | 0.687 | 0.731 | 0.792 |
| R4 additive refinement (30-epoch schedule) | 0.688 | 0.805 | (stopped) |

The fixed layers are matrix multiplications with constant matrices derived from $G$; all decisions are made by the
trained layers.


In [ ]:
VERSIONS36 = {
    # H2: transformer over the location tokens (width 128, 6 blocks, 8 heads), plus the value head; all four codes.
    "H2": dict(width=128, blocks=6, heads=8, lr=5e-4, aux=1.0, epochs={3: 60, 4: 120, 5: 150, 6: 240}),
    # H3: the same with width 192, for the two hardest codes.
    "H3": dict(width=192, blocks=6, heads=8, lr=4e-4, aux=1.0, epochs={5: 90, 6: 150}),
}


def recipe36(version, code):
    """The recipe of a version for one code (the number of epochs depends on the code); None if the version is not
    used for that code."""
    R = dict(VERSIONS36[version])
    if code not in R["epochs"]:
        return None
    R["epochs"] = EPOCHS if EPOCHS is not None else R["epochs"][code]
    return R


def transformer_block(h, width, heads, name):
    z = tf.keras.layers.LayerNormalization(name=f"{name}_attention_norm")(h)
    z = tf.keras.layers.MultiHeadAttention(heads, width // heads, name=f"{name}_attention")(z, z)
    h = tf.keras.layers.Add()([h, z])
    z = tf.keras.layers.LayerNormalization(name=f"{name}_ffn_norm")(h)
    z = tf.keras.layers.Dense(4 * width, activation="gelu", name=f"{name}_ffn_hidden")(z)
    return tf.keras.layers.Add()([h, tf.keras.layers.Dense(width, name=f"{name}_ffn_out")(z)])


def fixed_layer(equation, output_shape, kernel, name):
    """A linear layer with a constant (not trained) kernel."""
    layer = tf.keras.layers.EinsumDense(equation, output_shape=output_shape, name=name, trainable=False)
    layer.build((None, kernel.shape[0]))
    layer.set_weights([kernel])
    return layer


def build_network36(code, width, blocks, heads, aux=0.0, logits_only=False, **_):
    """Input: syndrome s (r numbers). Output: n logits, followed by n value estimates (e_j / Delta) when aux > 0
    (training), or only the logits with logits_only (final models)."""
    C = CODES[code]; n, r = C["n"], C["r"]
    inp = tf.keras.Input(shape=(r,), name="syndrome")
    # Fixed (not trained): token j = ROTATIONS[j] @ s, the syndrome seen from location j.
    x = fixed_layer("br,rjq->bjq", (n, r), np.transpose(C["ROTATIONS"], (2, 0, 1)), "frames")(inp)
    # Shared linear embedding of every token plus a learned per-location bias (location embedding).
    h = tf.keras.layers.EinsumDense("bjq,qd->bjd", output_shape=(n, width), bias_axes="jd", name="embed")(x)
    h = tf.keras.layers.Dense(width, activation="gelu", name="embed_mlp")(h)
    for b in range(blocks):
        h = transformer_block(h, width, heads, f"block{b}")
    h = tf.keras.layers.LayerNormalization(name="final_norm")(h)
    logits = tf.keras.layers.Reshape((n,), name="logits")(tf.keras.layers.Dense(1, name="token_logit")(h))
    if aux == 0 or logits_only:
        return tf.keras.Model(inp, logits, name=f"ume_locator_code{code}")
    value = tf.keras.layers.Reshape((n,), name="value")(tf.keras.layers.Dense(1, name="token_value")(h))
    return tf.keras.Model(inp, tf.keras.layers.Concatenate(name="logits_and_values")([logits, value]),
                          name=f"ume_locator_code{code}")


for version in VERSIONS36:
    for code in HARMONIC_A:
        if recipe36(version, code) is not None:
            net = build_network36(code, **recipe36(version, code))
            print(f"{version} code {code}: {net.count_params():,} parameters "
                  f"({sum(int(np.prod(w.shape)) for w in net.trainable_weights):,} trained), "
                  f"{recipe36(version, code)['epochs']} epochs")


## 13. Training (Codes 3 to 6)

- **Loss:** the masked binary cross-entropy of Part I on the logits (target 1 at required, 0 at forbidden locations,
  optional locations masked out), plus `aux` times the mean squared error of the value estimates ($e_j / \Delta$ at
  every location, averaged over the stages). The value heads make the network track every UME's value, which the
  refinement stages need.
- **Data:** every batch of 4,096 codewords is drawn anew by `tf_sample_harmonic` (float64 on the GPU), half Group I and
  half Group II, as in the description.
- **Optimiser, schedule and validation** as in Part I (AdamW, warm-up and cosine decay; after every epoch the first
  200,000 validation codewords are decoded and the epoch with the fewest errors is kept), with 1,000 steps per epoch
  and a number of epochs that grows with the difficulty of the code. Seed 636 + code.
- **Resumable:** the training state is backed up after every epoch (`tf.keras.callbacks.BackupAndRestore`), so an
  interrupted run continues where it stopped when the notebook is run again.


In [ ]:
VAL_EPOCH_SIZE = min(200_000, EVAL_SIZE)        # validation codewords checked after every epoch


def make_loss36(n, aux):
    def loss(y_true, y_pred):
        target, mask, value = y_true[:, :n], y_true[:, n:2 * n], y_true[:, 2 * n:]
        bce = tf.nn.sigmoid_cross_entropy_with_logits(labels=target, logits=y_pred[:, :n])
        total = tf.reduce_sum(bce * mask) / tf.maximum(tf.reduce_sum(mask), 1.0)
        if aux > 0:
            stages = y_pred.shape[1] // n - 1
            for t in range(stages):
                total += aux / stages * tf.reduce_mean(tf.square(y_pred[:, n * (t + 1):n * (t + 2)] - value))
        return total
    return loss


def make_training_batch36(code):
    Delta = CODES[code]["Delta"]

    def training_batch(_):
        s, e = tf_sample_harmonic(code, BATCH)
        required = tf.cast(tf.abs(e) > Delta, tf.float32)
        optional = tf.cast((e != 0) & (tf.abs(e) <= Delta), tf.float32)
        return s, tf.concat([required, 1.0 - optional, tf.cast(e / Delta, tf.float32)], axis=1)
    return training_batch


_LOGIT_FUNCTIONS = {}                                     # one compiled function per network (kept with the network)


def logits36(network, code, s):
    """The network's logits (float32, GPU if available) for syndromes s; the value outputs are dropped."""
    if id(network) not in _LOGIT_FUNCTIONS:
        _LOGIT_FUNCTIONS[id(network)] = (network, tf.function(lambda b: network(b, training=False), jit_compile=True))
    f = _LOGIT_FUNCTIONS[id(network)][1]
    out = np.concatenate([np.asarray(f(tf.constant(s[i:i + 8192], tf.float32))) for i in range(0, len(s), 8192)])
    return out[:, :CODES[code]["n"]]


class Validate36(tf.keras.callbacks.Callback):
    """Validation after every epoch; keeps the epoch with the fewest wrong codewords (ties: lower validation loss).
    The history is written after every epoch, so an interrupted run can resume it."""

    def __init__(self, code, recipe, s, e, g, path, history_path):
        super().__init__(); self.code, self.recipe, self.s, self.e, self.g = code, recipe, s, e, g
        self.path, self.history_path = path, history_path
        self.history = json.loads(history_path.read_text())["epochs"] if history_path.exists() else []
        self.best = min(((r["val_errors"], r["val_loss"]) for r in self.history), default=(np.inf, np.inf))
        self.start = time.time() - 60 * (self.history[-1]["minutes"] if self.history else 0)

    def on_epoch_end(self, epoch, logs=None):
        logits = logits36(self.model, self.code, self.s)
        ok = correct(self.code, self.e, logits > 0); errors = int((~ok).sum()); loss = bce_np(self.code, logits, self.e)
        lr = float(np.asarray(self.model.optimizer.learning_rate))
        row = dict(epoch=epoch + 1, loss=float(logs["loss"]), val_loss=loss, val_errors=errors,
                   val_errors_group1=int((~ok[self.g == 1]).sum()), val_errors_group2=int((~ok[self.g == 2]).sum()),
                   lr=lr, minutes=(time.time() - self.start) / 60)
        self.history = [r for r in self.history if r["epoch"] < epoch + 1] + [row]
        improved = (errors, loss) < self.best
        if improved:
            self.best = (errors, loss); self.model.save(self.path)
        self.history_path.write_text(json.dumps(dict(recipe=self.recipe, steps=STEPS36, batch=BATCH, epochs=self.history), indent=1))
        print(f"epoch {epoch + 1:3d}  loss {row['loss']:.3e}  val_loss {loss:.3e}  val_errors {errors:7d} "
              f"({errors / len(self.s):.2e}; Group I {row['val_errors_group1']}, II {row['val_errors_group2']})  "
              f"lr {lr:.2e}  {row['minutes']:.1f} min{'  *' if improved else ''}", flush=True)


def model_path36(version, code):
    return OUT / f"model_{version}_code{code}.keras"


def train_network36(version, code):
    R = recipe36(version, code); n = CODES[code]["n"]
    history_path = OUT / f"history_{version}_code{code}.json"
    backup_dir = OUT / f"backup_{version}_code{code}"
    done = len(json.loads(history_path.read_text())["epochs"]) if history_path.exists() else 0
    if done >= R["epochs"] and not backup_dir.exists():
        print(f"{version} code {code}: already trained ({R['epochs']} epochs)"); return
    # Seed 636 + code; a resumed run (after an interruption) draws a new stream of training batches.
    tf.keras.utils.set_random_seed(636 + code + (1009 * done if backup_dir.exists() else 0))
    network = build_network36(code, **R)
    steps = R["epochs"] * STEPS36; warmup = min(2000, steps // 10)
    lr = tf.keras.optimizers.schedules.CosineDecay(R["lr"] * 0.01, steps - warmup, alpha=0.01, warmup_target=R["lr"],
                                                   warmup_steps=warmup)
    network.compile(optimizer=tf.keras.optimizers.AdamW(lr, weight_decay=1e-5, beta_2=0.95, clipnorm=1.0),
                    loss=make_loss36(n, R["aux"]))
    print(f"{version} code {code}: {R} | {STEPS36} steps x {BATCH} codewords per epoch", flush=True)
    data = tf.data.Dataset.range(1).repeat().map(make_training_batch36(code), num_parallel_calls=tf.data.AUTOTUNE).prefetch(8)
    E = EVAL36[code]; m = VAL_EPOCH_SIZE                  # the first 200,000 validation codewords after every epoch
    val = Validate36(code, R, E["s_val"][:m], E["e_val"][:m], E["g_val"][:m], model_path36(version, code), history_path)
    backup = tf.keras.callbacks.BackupAndRestore(str(backup_dir))
    network.fit(data, epochs=R["epochs"], steps_per_epoch=STEPS36, verbose=0,
                callbacks=[val, backup, tf.keras.callbacks.TerminateOnNaN()])


NETWORKS36 = {}
for version in VERSIONS36_TO_RUN:
    for code in CODES_36:
        if recipe36(version, code) is None:              # this version is not used for this code
            continue
        if TRAIN:
            train_network36(version, code)
        if model_path36(version, code).exists():
            NETWORKS36[version, code] = tf.keras.models.load_model(model_path36(version, code), compile=False)
            print(f"Loaded {model_path36(version, code)}")
        else:
            print(f"{model_path36(version, code)} not found: train it with P3_TRAIN=1")


## 14. Training Curves (Codes 3 to 6)

Training loss (masked BCE plus the auxiliary value loss, on the current batches), validation BCE and the number of
wrongly decoded validation codewords (out of 1,000,000) after every epoch.


In [ ]:
HISTORIES36 = {}
for version in VERSIONS36_TO_RUN:
    for code in CODES_36:
        path = OUT / f"history_{version}_code{code}.json"
        if path.exists():
            HISTORIES36[version, code] = json.loads(path.read_text())["epochs"]

if HISTORIES36:
    fig, axes = plt.subplots(len(CODES_36), 2, figsize=(13, 4.2 * len(CODES_36)), squeeze=False)
    for row, code in enumerate(CODES_36):
        for version in VERSIONS36_TO_RUN:
            if (version, code) not in HISTORIES36:
                continue
            h = HISTORIES36[version, code]; epochs = [r["epoch"] for r in h]
            line, = axes[row, 0].plot(epochs, [r["loss"] for r in h], "--", alpha=0.7, label=f"{version} training (BCE + aux)")
            axes[row, 0].plot(epochs, [r["val_loss"] for r in h], color=line.get_color(), label=f"{version} validation BCE")
            axes[row, 1].plot(epochs, [max(r["val_errors"], 0.5) for r in h], "o-", ms=3, color=line.get_color(), label=version)
        axes[row, 0].set(yscale="log", xlabel="epoch", ylabel="loss", title=f"Code {code}: loss")
        axes[row, 1].set(yscale="log", xlabel="epoch", ylabel="wrong codewords (of 1M)", title=f"Code {code}: validation errors")
        for ax in axes[row]:
            ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()
    for (version, code), h in HISTORIES36.items():
        best = min(h, key=lambda r: (r["val_errors"], r["val_loss"]))
        print(f"{version} code {code}: {len(h)} epochs in {h[-1]['minutes']:.0f} min; best epoch {best['epoch']} "
              f"(val errors {best['val_errors']:,} = {best['val_errors'] / EVAL_SIZE:.2e}, val BCE {best['val_loss']:.2e})")


## 15. Validation and Test Performance (Codes 3 to 6)

**Symmetry averaging.** These codes have exact symmetries that map codewords to codewords, UMEs to UMEs at moved
locations, and leave the distribution of the noisy codewords unchanged:

- the negacyclic **shift** ($y_j \to y_{j-1}$, $y_0 \to -y_{n-1}$; location $j \to j + 1$),
- the **reflection** ($y_0 \to y_0$, $y_j \to -y_{n-j}$; location $j \to n - j$), since the cosines are even and the
  sines odd,
- the **sign** change $y \to -y$ (locations unchanged).

The networks are trained on every shift equally often, but nothing makes them exactly symmetric, and their errors on
the transformed copies of a codeword are partly independent. The final decoders therefore average the logits over
several transformed copies ("views"; each view's logits are moved back to the original locations). In the syndrome
domain every view is an orthogonal matrix $M_v = H V H^T$ ($V$ the signed permutation that transforms $y$), and
moving the logits back is a permutation, so a decoder is the same trained network applied to fixed linear transforms
of its input, with the outputs averaged. Tests on development networks showed that reflection and sign change help
much more than more shifts (code 6: one view 73.3%, four shifts 76.1%, the eight views below 78.3%).

- **Single networks:** 8 views, the shifts by $0$ and $n/2$, each with and without reflection, each with both signs.
- **Ensembles:** where two versions are trained (codes 5 and 6), the two networks together with 4 views each (with
  and without reflection, both signs), so that every candidate costs 8 network evaluations per codeword.

For every candidate the threshold is chosen on the validation set as in Part I, and the final model of a code is the
candidate with the fewest validation errors (ties: fewer parameters). The test set is decoded once. The test accuracy
of every single network *without* averaging is shown for comparison.


In [ ]:
VIEWS_SINGLE = [(reflect, shift, sign) for reflect in (False, True) for shift in (0, 1 / 2) for sign in (1, -1)]
VIEWS_PAIR = [(reflect, 0, sign) for reflect in (False, True) for sign in (1, -1)]
# Ensembles of the m best single networks with 8 / m views each (VIEWS_PAIR within VIEWS_SINGLE).
ENSEMBLES = {"pair": (2, VIEWS_PAIR)}


def view_matrices(code, views):
    """For each view (reflect, shift as a fraction of n, sign): M (r x r) with syndrome(view of y) = s M, and the
    location map loc (original location i is location loc[i] of the view)."""
    C = CODES[code]; n, H = C["n"], C["H"]
    reflection = np.zeros((n, n)); reflection[0, 0] = 1.0; reflection[n - np.arange(1, n), np.arange(1, n)] = -1.0
    out = []
    for reflect, shift, sign in views:
        k = round(shift * n)
        P = np.zeros((n, n)); P[np.arange(n - k), np.arange(k, n)] = 1.0; P[np.arange(n - k, n), np.arange(k)] = -1.0
        V = sign * ((reflection if reflect else np.eye(n)) @ P)              # y -> y V
        loc = (((n - np.arange(n)) % n if reflect else np.arange(n)) + k) % n
        out.append((H @ V @ H.T, loc))
    return out


def averaged_logits36(networks, code, s, views):
    """Logits averaged over the networks and the views, each view's logits moved back to the original locations."""
    total = 0.0
    for M, loc in view_matrices(code, views):
        sv = (s.astype(np.float64) @ M).astype(np.float32)
        for network in networks:
            total = total + logits36(network, code, sv)[:, loc]
    return total / (len(networks) * len(views))


def choose_threshold(code, e, logits):
    errs = np.array([(~correct(code, e, logits > t)).sum() for t in GRID])
    best = GRID[errs == errs.min()]
    return float(best[len(best) // 2]), int(errs.min())


def view_logits36(version, code):
    """For the validation and test sets: the network's logits without averaging (the first view is the identity), and
    its averages over all VIEWS_SINGLE ("single") and over the views of each ensemble ("pair")."""
    E, net = EVAL36[code], NETWORKS36[version, code]
    groups = {"single": VIEWS_SINGLE, **{key: views for key, (m, views) in ENSEMBLES.items()}}
    out = {}
    for split in ("val", "test"):
        sums = {key: 0.0 for key in groups}
        for view, (M, loc) in zip(VIEWS_SINGLE, view_matrices(code, VIEWS_SINGLE)):
            lv = logits36(net, code, (E[f"s_{split}"].astype(np.float64) @ M).astype(np.float32))[:, loc]
            if view == (False, 0, 1):
                out[split, "plain"] = lv
            for key, views in groups.items():
                if view in views:
                    sums[key] = sums[key] + lv
        for key, views in groups.items():
            out[split, key] = sums[key] / len(views)
    return out


def summarize36(name, code, versions, views, lv, lt, plain=None):
    E = EVAL36[code]; ev, et, gt = E["e_val"], E["e_test"], E["g_test"]
    t, val_errors = choose_threshold(code, ev, lv)
    rep = lt > t; ok = correct(code, et, rep)
    test_plain = None
    if plain is not None:                        # the same network without averaging, for comparison
        t1, _ = choose_threshold(code, ev, plain[0])
        test_plain = float(correct(code, et, plain[1] > t1).mean())
    return dict(
        name=name, versions=versions, views=views, code=code,
        parameters=sum(NETWORKS36[v, code].count_params() for v in versions),
        threshold=t, val_errors=val_errors, val_accuracy=1 - val_errors / len(ev), test_errors=int((~ok).sum()),
        test_accuracy=float(ok.mean()), test_accuracy_plain=test_plain,
        test_accuracy_group1=float(ok[gt == 1].mean()), test_accuracy_group2=float(ok[gt == 2].mean()),
        test_missed_required=int(((np.abs(et) > CODES[code]["Delta"]) & ~rep).any(1).sum()),
        test_forbidden_reported=int((rep & (et == 0)).any(1).sum()))


# Every single network (8 views), then (codes with enough versions) the ensembles of the m best ones on the validation
# set (8 / m views each; their averages over those views are kept from the single-network pass).
RESULTS36, ENSEMBLE_LOGITS = {}, {}
for (version, code) in NETWORKS36:
    L = view_logits36(version, code)
    RESULTS36[version, code] = summarize36(version, code, [version], VIEWS_SINGLE, L["val", "single"], L["test", "single"],
                                           plain=(L["val", "plain"], L["test", "plain"]))
    for key in ENSEMBLES:
        ENSEMBLE_LOGITS[version, code, key] = (L["val", key], L["test", key])
    del L
for code in CODES_36:
    singles = sorted((r for (v, c), r in RESULTS36.items() if c == code), key=lambda r: (r["val_errors"], r["parameters"]))
    for key, (m, views) in ENSEMBLES.items():
        if len(singles) >= m:
            names = [r["name"] for r in singles[:m]]
            lv = sum(ENSEMBLE_LOGITS[v, code, key][0] for v in names) / m
            lt = sum(ENSEMBLE_LOGITS[v, code, key][1] for v in names) / m
            RESULTS36["+".join(names), code] = summarize36("+".join(names), code, names, views, lv, lt)
del ENSEMBLE_LOGITS

label_width = max(len(f"{name} code {code}") for name, code in RESULTS36) + 1 if RESULTS36 else 16
print(f"{'':{label_width}s} {'views':>5s} {'params':>10s} {'threshold':>9s} {'val accuracy':>12s} {'test accuracy':>13s} {'(no averaging)':>14s} "
      f"{'Group I':>9s} {'Group II':>9s} {'missed required':>15s} {'forbidden reported':>18s}")
for (name, code), r in sorted(RESULTS36.items(), key=lambda kv: (kv[0][1], len(kv[1]["versions"]), kv[0][0])):
    plain = "" if r["test_accuracy_plain"] is None else f"{r['test_accuracy_plain']:.6f}"
    print(f"{name + ' code ' + str(code):{label_width}s} {len(r['views']) * len(r['versions']):5d} {r['parameters']:10,d} {r['threshold']:9.2f} "
          f"{r['val_accuracy']:12.6f} {r['test_accuracy']:13.6f} {plain:>14s} {r['test_accuracy_group1']:9.6f} "
          f"{r['test_accuracy_group2']:9.6f} {r['test_missed_required']:15,d} {r['test_forbidden_reported']:18,d}")
print(f"({EVAL_SIZE:,} validation and test codewords per code; views = network evaluations per codeword; missed "
      "required / forbidden reported: test codewords with at least one such location, a codeword can have both)")

FINAL36 = {}
for code in CODES_36:
    candidates = [r for (name, c), r in RESULTS36.items() if c == code]
    if candidates:
        FINAL36[code] = min(candidates, key=lambda r: (r["val_errors"], r["parameters"]))
        print(f"Code {code}: final model {FINAL36[code]['name']} with {len(FINAL36[code]['views'])} views per network "
              f"(threshold {FINAL36[code]['threshold']:g}, test accuracy {FINAL36[code]['test_accuracy']:.6f})")


## 16. The Final Models (Codes 3 to 6)

As in Part I (threshold folded in, float64), with the symmetry averaging built in. The final model takes $y$ ($n$
numbers), computes the syndrome $s = yH^T / \Delta$ (a constant float64 matrix multiplication), applies the view
matrices $M_v$ (constant matrix multiplications), runs the chosen network(s) on each view (shared weights), moves each
output back to the original locations (a constant permutation matrix), averages them and subtracts the threshold. Only the
logits are used (no value heads). The cell saves `final_model_v4_code3.keras` to `final_model_v4_code6.keras`,
checks on the first 20,000 test codewords that the final models decode like the float32 networks, compares them with
the provided example outputs (which include some optional locations as well), and runs them on these 20,000 test
codewords written in the format of the test files. These models run on a GPU when one is available (float64 either
way), because the averaging multiplies the work by eight; like the final cell, the checks decode them in batches of
1,024 (codes 3 and 4), 512 (code 5) and 256 (code 6) codewords, so that they also fit smaller GPUs.


In [ ]:
FINAL_CHECK_36 = 20_000


def make_final_model36(code, r):
    C = CODES[code]; n = C["n"]
    tf.keras.config.set_dtype_policy("float64")
    try:
        nets = [build_network36(code, **recipe36(v, code), logits_only=True) for v in r["versions"]]
        named = [tf.keras.Model(net.input, net.output, name=f"{v}_code{code}") for v, net in zip(r["versions"], nets)]
        inp = tf.keras.Input(shape=(n,), dtype="float64", name="y")   # y is not rounded to float32
        s = fixed_layer("bi,iq->bq", (C["r"],), C["H"].T / C["Delta"], "syndrome")(inp)
        outputs = []
        for v, (M, loc) in enumerate(view_matrices(code, r["views"])):
            sv = fixed_layer("bq,qp->bp", (C["r"],), M, f"view_{v}")(s)
            back = np.zeros((n, n)); back[loc, np.arange(n)] = 1.0          # view location loc[i] -> location i
            for net in named:
                outputs.append(fixed_layer("bj,ji->bi", (n,), back, f"view_{v}_back_{net.name}")(net(sv)))
        mean = tf.keras.layers.Average(name="average")(outputs)
        out = tf.keras.layers.Rescaling(1.0, offset=-r["threshold"], name="minus_threshold")(mean)
        final = tf.keras.Model(inp, out, name=f"final_ume_locator_code{code}")
    finally:
        tf.keras.config.set_dtype_policy("float32")
    for v, net in zip(r["versions"], nets):         # copy the trained weights layer by layer (no value heads)
        trained = {layer.name: layer for layer in NETWORKS36[v, code].layers}
        for layer in net.layers:
            if layer.weights:
                layer.set_weights([w.astype(np.float64) for w in trained[layer.name].get_weights()])
    return final


FINAL_BATCH = {1: 4096, 2: 4096, 3: 1024, 4: 1024, 5: 512, 6: 256}     # codewords per call, as in the final cell


def final_outputs(final, y, code):
    """As the final cell: the final model's float64 outputs, in batches (GPU if available)."""
    b = FINAL_BATCH[code]
    out = [np.asarray(final(tf.constant(y[i:i + b], tf.float64), training=False)) for i in range(0, len(y), b)]
    return np.concatenate(out) if out else np.zeros((0, CODES[code]["n"]))


def decode36(final, y, code):
    """As the final cell: location reported when the final model's output is positive."""
    return final_outputs(final, y, code) > 0


for code, r in FINAL36.items():
    final = make_final_model36(code, r)
    final.save(final_model_path(code))
    final = tf.keras.models.load_model(final_model_path(code), compile=False)
    E = EVAL36[code]; m = FINAL_CHECK_36; y, e = E["y_test"][:m], E["e_test"][:m]
    start = time.time(); rep64 = decode36(final, y, code); seconds = time.time() - start
    rep32 = averaged_logits36([NETWORKS36[v, code] for v in r["versions"]], code, E["s_test"][:m], r["views"]) > r["threshold"]
    print(f"Code {code}: saved {final_model_path(code)} ({r['name']}, threshold {r['threshold']:g}, {len(final.layers)} layers); "
          f"accuracy on the first {m:,} test codewords {correct(code, e, rep64).mean():.6f} (float32 networks "
          f"{correct(code, e, rep32).mean():.6f}); decoded differently: {(rep64 != rep32).any(1).sum()}; "
          f"{m / seconds:,.0f} codewords/s")
    if code in EXAMPLES:
        X, S = EXAMPLES[code]
        ours = to_sets(decode36(final, X, code)); check_output(ours, len(X), CODES[code]["n"])
        print(f"  examples: identical to the example output {np.mean([a == b for a, b in zip(ours, S)]):.3f}, "
              f"subset of it {np.mean([a <= b for a, b in zip(ours, S)]):.3f}, superset {np.mean([a >= b for a, b in zip(ours, S)]):.3f}")
    heldout = OUT / f"Noisy-Codeword-Heldout-for-Code-{code}.pkl"
    with open(heldout, "wb") as file:
        pickle.dump([[float(v) for v in row] for row in y], file)
    X = np.asarray(load_pickle(heldout), dtype=np.float64)
    ours = to_sets(decode36(final, X, code)); check_output(ours, len(X), CODES[code]["n"])
    reported = np.zeros((len(X), CODES[code]["n"]), bool)
    for row, s in enumerate(ours):
        reported[row, list(s)] = True
    print(f"  {heldout.name}: {len(ours):,} outputs pass the format check, accuracy {correct(code, e, reported).mean():.6f}")


### 16.1 One File for the Six Final Models

The final models of the six codes (sections 9 and 16) are stored together in one file, `final_model_v4.keras`: a
Keras model whose six parts are the unchanged final models, named `code1` to `code6` (one input and one output per
code). The final cell loads this one file and takes the part of each code by its number (`get_layer("code3")`, for
instance), never by the input length. The cell reloads the file and checks, on the held-out codewords of every code,
that each part gives exactly the same float64 outputs as the separately saved final model.


In [ ]:
ALL_CODES = [1, 2, 3, 4, 5, 6]


def bundle_path():
    return OUT / f"final_model_{NOTEBOOK_TAG}.keras"


if sorted(CODES_TO_RUN) == ALL_CODES and all(final_model_path(c).exists() for c in ALL_CODES):
    finals = {c: tf.keras.models.load_model(final_model_path(c), compile=False) for c in ALL_CODES}
    inputs = [tf.keras.Input(shape=(CODES[c]["n"],), dtype="float64", name=f"y_code{c}") for c in ALL_CODES]
    parts = [tf.keras.Model(finals[c].inputs, finals[c].outputs, name=f"code{c}") for c in ALL_CODES]
    bundle = tf.keras.Model(inputs, [part(x) for part, x in zip(parts, inputs)], name=f"final_ume_locators_{NOTEBOOK_TAG}")
    bundle.save(bundle_path())
    bundle = tf.keras.models.load_model(bundle_path(), compile=False)
    print(f"Saved {bundle_path()} ({bundle_path().stat().st_size / 1e6:.1f} MB): parts "
          f"{[layer.name for layer in bundle.layers if isinstance(layer, tf.keras.Model)]}")
    for c in ALL_CODES:
        X = np.asarray(load_pickle(OUT / f"Noisy-Codeword-Heldout-for-Code-{c}.pkl"), dtype=np.float64)[:FINAL_CHECK_36]
        separate, part = final_outputs(finals[c], X, c), final_outputs(bundle.get_layer(f"code{c}"), X, c)
        print(f"Code {c}: part code{c} on {len(X):,} held-out codewords: outputs identical to {final_model_path(c).name}: "
              f"{np.array_equal(separate, part)}; decisions identical: {np.array_equal(separate > 0, part > 0)}")
    del finals, parts, bundle
else:
    print("One-file bundle skipped: it needs the final models of all six codes from this run.")


## 17. Final Cell: Decode the Six Test Files

This cell loads the one model file `final_model_v4.keras` (the six final models, section
16.1; from the notebook folder or `artifacts/submission/`), reads `Noisy-Codeword-Test-for-Code-1.pkl` to
`Noisy-Codeword-Test-for-Code-6.pkl` (lists of lists of $n$ real numbers), decodes every codeword in float64 (on the
GPU if one is available, otherwise on the CPU) and writes
`Decode-Results-Test-for-Code-1.pkl` to `Decode-Results-Test-for-Code-6.pkl` (lists of sets of integer locations, in
the input order) with `pickle.dump`. The model of each code is selected by the code number (`get_layer("code3")`, for
instance), not by the input length (codes 2 and 3 both have $n = 16$). The codewords are decoded in batches whose
size depends on the code (halved if a GPU runs out of memory; this does not change any decision). The cell performs
no training. A missing input file is skipped with a message.


In [ ]:
# ===== Final cell: load the trained DNNs and decode the six test files =====
# On Google Colab, first install the package versions used for training, so the saved networks load and
# decode identically (elsewhere the project environment is used).
import sys

if "google.colab" in sys.modules:
    %pip install -q "numpy>=2,<3" "tensorflow==2.21.0" "keras==3.15.1"

import pickle
from pathlib import Path

import numpy as np
import tensorflow as tf

if "google.colab" in sys.modules and tf.__version__ != "2.21.0":
    raise RuntimeError(f"TensorFlow {tf.__version__} was already loaded: restart the runtime (Runtime > Restart session) and run this cell again.")

CODE_LENGTHS = {1: 10, 2: 16, 3: 16, 4: 20, 5: 40, 6: 80}   # code number -> codeword length n
MODEL_FILE = "final_model_v4.keras"   # the six final models in one file, as the parts "code1" to "code6"
MODEL_FOLDERS = [".", "artifacts/submission"]
BATCH = {1: 4096, 2: 4096, 3: 1024, 4: 1024, 5: 512, 6: 256}   # codewords per call (halved if the GPU runs out of memory)
DEVICE = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"   # float64 on either device


class CompatUnpickler(pickle.Unpickler):
    # Pickles written with NumPy 2 reference numpy._core; NumPy 1.x only has numpy.core.
    def find_class(self, module, name):
        if int(np.__version__.split(".")[0]) < 2 and module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)


models, model_file = None, None
for code, n in CODE_LENGTHS.items():
    input_path = Path(f"Noisy-Codeword-Test-for-Code-{code}.pkl")
    output_path = Path(f"Decode-Results-Test-for-Code-{code}.pkl")
    if not input_path.exists():
        print(f"Code {code}: {input_path} not found, skipped"); continue
    if models is None:                               # loaded once, when the first test file is found
        model_file = next((Path(f) / MODEL_FILE for f in MODEL_FOLDERS if (Path(f) / MODEL_FILE).exists()), None)
        if model_file is None:
            raise FileNotFoundError(f"{MODEL_FILE} not found in {MODEL_FOLDERS}")
        with tf.device(DEVICE):
            models = tf.keras.models.load_model(model_file, compile=False)
    network = models.get_layer(f"code{code}")        # the final model of this code, selected by the code number
    if network.inputs[0].shape[-1] != n:
        raise ValueError(f"Code {code}: the part code{code} of {model_file} expects {network.inputs[0].shape[-1]} numbers, not {n}")

    with open(input_path, "rb") as file:
        samples = CompatUnpickler(file).load()
    y = np.asarray(samples, dtype=np.float64).reshape(len(samples), -1) if len(samples) else np.zeros((0, n))
    if y.shape[1] != n or not np.isfinite(y).all():
        raise ValueError(f"Code {code}: expected {n} finite real numbers per codeword, got shape {y.shape}")

    # Decoding: the network's outputs (float64, threshold folded in); location j is reported when output j > 0.
    reported = np.zeros(y.shape, dtype=bool)
    batch, start = BATCH[code], 0
    with tf.device(DEVICE):
        while start < len(y):
            try:
                outputs = network(tf.constant(y[start:start + batch], tf.float64), training=False)
            except tf.errors.ResourceExhaustedError:      # GPU out of memory: smaller batches, same decisions
                if batch == 1:
                    raise
                batch //= 2
                continue
            reported[start:start + batch] = np.asarray(outputs) > 0
            start += batch
    results = [set(int(j) for j in np.flatnonzero(row)) for row in reported]

    # Format check: a list with one set of int locations in [0, n) per input codeword, in input order.
    assert type(results) is list and len(results) == len(samples)
    assert all(type(s) is set and all(type(j) is int and 0 <= j < n for j in s) for s in results)
    with open(output_path, "wb") as file:
        pickle.dump(results, file)
    print(f"Code {code}: {model_file} (part code{code}) decoded {len(results):,} codewords on {DEVICE} -> {output_path} "
          f"(mean set size {np.mean([len(s) for s in results]) if results else 0:.3f}; first five {results[:5]})")
